# Module 9 Practical: Clustering

This notebook combines both Module 9 learning activities:

- **Part 1:** Determining a suitable value of K using the elbow method.
- **Part 2:** Comparing K-means clustering results using the Iris dataset.


## 1. Import the required libraries

- `KMeans` performs K-means clustering.
- `StandardScaler` puts income and age on a similar scale.
- `matplotlib` creates graphs.
- `numpy` stores the data.
- `pandas` helps us display the elbow results.


In [ ]:
!pip install scikit-learn matplotlib pandas -q

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random


## 2. Create a small example dataset

The function below creates five groups of customers. Each group has a different average income and age.

`random.seed()` and `np.random.seed()` make the example reproducible, so the same data is created each time.


In [ ]:
random.seed(42)
np.random.seed(42)

def create_clustered_data(number_of_people, number_of_groups):
    people_per_group = number_of_people // number_of_groups
    data = []

    for group in range(number_of_groups):
        income_centre = random.uniform(20000, 200000)
        age_centre = random.uniform(20, 70)

        for person in range(people_per_group):
            income = np.random.normal(income_centre, 8000)
            age = np.random.normal(age_centre, 1.5)
            data.append([income, age])

    return np.array(data)

data = create_clustered_data(100, 5)

print(data[:5])


## 3. View the original data

Each point represents one customer:

- the x-axis shows income;
- the y-axis shows age.

The visible groups give us an initial idea that the data may contain several clusters.


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(data[:, 0], data[:, 1])

plt.xlabel("Income")
plt.ylabel("Age")
plt.title("Example Customer Data")
plt.show()


## 4. Scale the variables

Income is measured in thousands of dollars, while age is measured in years. Without scaling, income would have a much larger influence on the distance calculation.

`StandardScaler()` changes each variable so that it has a similar scale.


In [ ]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(data[:, 0], data[:, 1])
axes[0].set_title("Original Data")
axes[0].set_xlabel("Income")
axes[0].set_ylabel("Age")

axes[1].scatter(scaled_data[:, 0], scaled_data[:, 1])
axes[1].set_title("Scaled Data")
axes[1].set_xlabel("Scaled Income")
axes[1].set_ylabel("Scaled Age")

plt.tight_layout()
plt.show()


## 5. Calculate inertia for different values of K

**Inertia** is the total squared distance between each point and the centre of its assigned cluster.

A smaller inertia means that points are closer to their cluster centres. However, inertia always decreases as K increases, so we look for the point where the improvement starts to slow down.

This point is called the **elbow**.


In [ ]:
inertia_values = []

for k in range(1, 11):
    model = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=42
    )

    model.fit(scaled_data)
    inertia_values.append(model.inertia_)


The table below shows the inertia calculated for each value of K.


In [ ]:
elbow_results = pd.DataFrame({
    "K": range(1, 11),
    "Inertia": inertia_values
})

elbow_results


## 6. Draw the elbow plot

Look for a bend in the line. Before the bend, adding clusters produces a large improvement. After the bend, the improvement becomes much smaller.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    elbow_results["K"],
    elbow_results["Inertia"],
    marker="o"
)

plt.xticks(range(1, 11))
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()


For this example, the elbow should appear around **K = 5**.

The elbow method is a guide rather than a strict rule. In a real project, we should also consider:

- whether the clusters are easy to explain;
- whether the clusters are useful for the business problem; and
- whether similar results appear when the model is run again.


## 7. Build the final K-means model

We now fit K-means using `K = 5`.

`fit_predict()` both fits the model and returns the cluster assigned to each customer.


In [ ]:
final_model = KMeans(
    n_clusters=5,
    n_init=10,
    random_state=42
)

cluster_labels = final_model.fit_predict(scaled_data)

print(cluster_labels)


## 8. Display the final clusters

The colour of each point shows the cluster assigned by K-means.

Cluster numbers such as 0, 1 and 2 are only labels. They do not indicate that one cluster is better or larger than another.


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    data[:, 0],
    data[:, 1],
    c=cluster_labels
)

plt.xlabel("Income")
plt.ylabel("Age")
plt.title("Customer Groups with K = 5")
plt.show()


## Discussion forum guide

A simple step-by-step answer could explain that you would:

1. Select suitable numeric features.
2. Clean missing or incorrect values.
3. Scale variables measured in different units.
4. Run K-means using several values of K.
5. Record the inertia for each K.
6. Draw an elbow plot.
7. Look for the point where further improvement becomes small.
8. Check whether the selected clusters are meaningful and useful.


---

# Part 2: Iris K-means Clustering

In this section, we compare clustering results using 8 clusters, 3 clusters, a poor starting point, and the actual Iris species.


## 1. Import the required libraries

The Iris dataset is included in Scikit-learn, so no separate CSV file is required.


In [ ]:
!pip install scikit-learn matplotlib -q

import matplotlib.pyplot as plt
import numpy as np

from sklearn.datasets import load_iris
from sklearn.cluster import KMeans


## 2. Load the Iris dataset

The dataset contains 150 flowers and four measurements:

- sepal length;
- sepal width;
- petal length; and
- petal width.

`X` contains the measurements used by K-means.

`y` contains the actual species labels. K-means does not use `y` when creating clusters. We only use it later for comparison.


In [ ]:
iris = load_iris()

X = iris.data
y = iris.target

print("Number of rows:", X.shape[0])
print("Number of variables:", X.shape[1])
print(iris.feature_names)


## 3. Create three K-means models

The models below demonstrate three situations:

- `8 clusters`: K is larger than the number of actual species.
- `3 clusters`: K matches the number of actual species.
- `3 clusters, poor start`: K is correct, but the initial cluster centres are less suitable.

`n_init=10` means that the normal model tries ten starting positions and keeps the best result.

The poor-start model uses only one random starting position to demonstrate why initialization can matter.


In [ ]:
models = [
    (
        "8 clusters",
        KMeans(
            n_clusters=8,
            n_init=10,
            random_state=42
        )
    ),
    (
        "3 clusters",
        KMeans(
            n_clusters=3,
            n_init=10,
            random_state=42
        )
    ),
    (
        "3 clusters, poor start",
        KMeans(
            n_clusters=3,
            init="random",
            n_init=1,
            random_state=43
        )
    )
]


## 4. Draw the four graphs

The graphs use three Iris measurements:

- petal width;
- sepal length; and
- petal length.

The first three graphs show K-means results. The final graph shows the actual species.


In [ ]:
fig = plt.figure(figsize=(11, 9))

for position, (title, model) in enumerate(models, start=1):
    model.fit(X)
    cluster_labels = model.labels_

    axis = fig.add_subplot(
        2,
        2,
        position,
        projection="3d",
        elev=48,
        azim=134
    )

    axis.scatter(
        X[:, 3],
        X[:, 0],
        X[:, 2],
        c=cluster_labels,
        edgecolor="black"
    )

    axis.set_xlabel("Petal width")
    axis.set_ylabel("Sepal length")
    axis.set_zlabel("Petal length")
    axis.set_title(title)

# Fourth graph: actual Iris species
axis = fig.add_subplot(
    2,
    2,
    4,
    projection="3d",
    elev=48,
    azim=134
)

axis.scatter(
    X[:, 3],
    X[:, 0],
    X[:, 2],
    c=y,
    edgecolor="black"
)

axis.set_xlabel("Petal width")
axis.set_ylabel("Sepal length")
axis.set_zlabel("Petal length")
axis.set_title("Actual Iris species")

plt.tight_layout()
plt.show()


## 5. How to explain the four graphs

### Graph 1: Eight clusters

The model divides the flowers into eight small groups. This is more than the three known Iris species, so some natural groups are divided into several smaller clusters.

This demonstrates **over-clustering**: K is too large for the main structure of the data.

### Graph 2: Three clusters

The model creates three broad groups. One group is clearly separated, while the other two overlap more.

The result is reasonably similar to the actual species pattern, although it is not identical.

### Graph 3: Three clusters with a poor starting point

The model still uses three clusters, but it begins from less suitable random cluster centres.

This can produce a weaker grouping because K-means may settle on a local solution. Running the algorithm several times with different starting positions reduces this risk.

### Graph 4: Actual Iris species

This graph uses the known species labels rather than K-means labels. It provides a reference for comparing the clustering results.

K-means cluster numbers do not automatically correspond to species names. For example, cluster `0` is not guaranteed to be Setosa.


## Comparison: Eight clusters versus three clusters

With **eight clusters**, the model splits the dataset into many small groups. The result is more detailed but less useful for representing the three main species.

With **three clusters**, the grouping is simpler and more closely resembles the actual species structure.

This comparison shows why choosing K is important. A value that is too large can divide meaningful groups into unnecessary smaller clusters.


## Discussion forum guide

In your response, describe:

- what each graph represents;
- how the eight-cluster result differs from the three-cluster result;
- why the three-cluster result is closer to the known Iris structure;
- how poor initialization can affect K-means; and
- why cluster labels should not be treated as actual class names.
